# Validation 01 — how fine does the *input* wavelength grid need to be?

The production catalog stores each source's SED on a **2 Å grid** (0.9–2.0 µm), and the disperser deposits one PSF stamp per input wavelength sample. So the cost of a dispersion is **linear in the number of input samples**, i.e. inversely proportional to the grid spacing: halving the spacing doubles the work. That makes the input spacing a speed knob — and the question for this notebook is how hard we can turn it before the **output image** degrades.

We treat this purely in **image space**: every input grid produces a dispersed image on the *same* 4088×4088 detector grid, so we just compare images pixel-for-pixel. (No 1D extraction enters — that would introduce a second, irrelevant output grid and its own sampling artifacts.) Concretely, we **sweep** the input spacing over 8 / 4 / 2 / 1 / 0.5 Å, disperse the same point source at each, and compare every image against a fine **0.25 Å reference** that is far finer than any tested spacing.

The metric is the relative L1 image difference,

$$\mathrm{L1} \;=\; \frac{\sum_{\mathrm{pix}} |\,I_{\Delta\lambda} - I_{\mathrm{ref}}\,|}{\sum_{\mathrm{pix}} I_{\mathrm{ref}}},$$

together with the total-flux error (how much the deposited counts drift). We run four spectra spanning the relevant regimes: a smooth **continuum**, two **synthetic emission lines** (FWHM 2 Å and 5 Å), and a **physical** KC96 starburst line. *Units:* µm for optics/PSF/disperser, Å for synphot/sensitivity; FLAM is erg s⁻¹ cm⁻² Å⁻¹; counts are e⁻ s⁻¹ per wavelength bin.

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("JAX_COMPILATION_CACHE_DIR",
                      str(Path.home() / ".cache" / "roman_grs_jax"))

# tutorial_helpers.py lives in the notebooks/ root, one level up from validation/.
nb_root = Path.cwd()
if not (nb_root / "tutorial_helpers.py").exists():
    nb_root = nb_root.parent
sys.path.insert(0, str(nb_root))

import time
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

from roman_disperser import paths, psf_model, star_disperser
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

print("JAX backend:", jax.default_backend())
print("reference data:", paths.data_dir())

SCA = 5
DETECTOR = f"WFI{SCA:02d}"
X, Y = 2000.0, 2000.0      # fixed source pixel position (1-indexed)

## 1. Set up the order-1 disperser

Standard wiring (as in notebook 02): optical payload, PSF payload, compiled star disperser. Everything here is fixed; only the input wavelength grid varies. Every spectrum is dispersed as a **point source**, so the trace and PSF are common to all cases and the grid is the single variable.

In [ ]:
model = RomanOpticalModel(config_file=str(paths.optical_model_path()))
opt1 = omj.make_sca_payload(model, sca=SCA, order="1")
psf1 = psf_model.get_or_make_psf_payload(
    detector=DETECTOR, order="1",
    cache_dir=str(paths.psf_cache_dir()), verbose=False,
)
disperse1 = star_disperser.make_star_disperser(psf1, opt1)
print("PSF grid:", psf1["psf_grid"].shape, "| oversample:", psf1["oversample"])

_out = jnp.zeros((4088, 4088), dtype=jnp.float32)
def disperse_counts(wl_um, counts):
    img = disperse1(X, Y, jnp.asarray(wl_um), jnp.asarray(counts), _out)
    img.block_until_ready()
    return np.asarray(img)

## 2. The four test spectra

Each "case" is a function that builds a count-rate spectrum on whatever grid it is handed, so the *same* spectrum can be sampled at any spacing:

- **continuum (g0v)** — a smooth stellar template, the production-typical case;
- **line FWHM 2 Å / 5 Å** — a faint flat continuum plus a Gaussian emission line at 1.5 µm, normalized to unit area so its integrated flux is fixed at `F_LINE`; widths at and above the production grid scale;
- **KC96 starburst (z=1.2)** — the bundled Kinney–Calzetti starburst template redshifted so Hα falls near 1.44 µm (a *physical* line, intrinsically ~25 Å wide because the template is natively ~11 Å).

The line center sits on a grid point at every spacing tested, so what we measure is the effect of *under-resolving the line width*, not an accidental peak-straddling miss (the sign of the flux error would flip if the line fell between samples; the magnitude is the point).

In [ ]:
LAM0_A = 15000.0           # synthetic line center, Å (1.5 µm; on every grid point)
F_LINE = 1.0e-16           # total synthetic line flux, erg/s/cm^2
F_CONT = 2.0e-19           # flat continuum FLAM, erg/s/cm^2/Å

def counts_continuum(wl_um, wl_a, dlam_a):
    _, c = th.template_to_counts("g0v", mag=16.0, sca=SCA, order="1", wl_um=wl_um)
    return c

def make_line(fwhm_a):
    sigma_a = fwhm_a / 2.3548
    def build(wl_um, wl_a, dlam_a):
        gauss = np.exp(-0.5 * ((wl_a - LAM0_A) / sigma_a) ** 2) / (sigma_a * np.sqrt(2*np.pi))
        flam = F_CONT + F_LINE * gauss
        sens = th.load_sensitivity(SCA, "1", wl_a)
        return flam * sens * dlam_a
    return build

def counts_kc96(wl_um, wl_a, dlam_a):
    _, c = th.template_to_counts("kc96_starb1", mag=18.0, sca=SCA, order="1",
                                 wl_um=wl_um, redshift=1.2)
    return c

CASES = {
    "continuum (g0v)":       counts_continuum,
    "line FWHM 2 Å":         make_line(2.0),
    "line FWHM 5 Å":         make_line(5.0),
    "KC96 starburst z=1.2":  counts_kc96,
}
SPACINGS = [8.0, 4.0, 2.0, 1.0, 0.5]     # Å, input grid spacings to test
REF_DLAM = 0.25                          # Å, fine reference (finer than all tested)

## 3. Run the sweep

For each spectrum we build the 0.25 Å reference image once, then disperse at each test spacing and compare to that reference. We record the relative L1 image difference, the total-flux error (which exposes line-flux mis-integration), the worst single-pixel fractional difference, and the sample count — the latter being the cost proxy, since the disperser runs linearly in samples (we confirm that in §5). Production spacing (2 Å) is flagged in the tables.

In [ ]:
def grid(dlam_a):
    return th.grism_wavelength_grid(dlam_angstrom=dlam_a)   # (wl_um, wl_a, dlam_a)

n_ref = grid(REF_DLAM)[0].size
results = {}
for name, build in CASES.items():
    wlr_um, wlr_a, dlr = grid(REF_DLAM)
    ref = disperse_counts(wlr_um, build(wlr_um, wlr_a, dlr))
    rsum = ref.sum()
    rows = []
    for s in SPACINGS:
        wl_um, wl_a, dl = grid(s)
        img = disperse_counts(wl_um, build(wl_um, wl_a, dl))
        d = img - ref
        sig = ref > 1e-3 * ref.max()
        rows.append(dict(
            spacing=s, n=wl_um.size, cost=wl_um.size / grid(2.0)[0].size,
            l1=np.abs(d).sum() / rsum,
            flux=(img.sum() - rsum) / rsum,
            maxpix=np.abs(d[sig] / ref[sig]).max(),
        ))
    results[name] = rows

for name, rows in results.items():
    print(f"\n{name}   (reference: {REF_DLAM} Å, {n_ref} samples)")
    print(f"  {'Δλ[Å]':>6} {'samples':>8} {'cost×':>6} {'L1[%]':>8} "
          f"{'flux err[%]':>11} {'max pix[%]':>10}")
    for r in rows:
        tag = "  <-- production" if r["spacing"] == 2.0 else ""
        print(f"  {r['spacing']:6.2f} {r['n']:8d} {r['cost']:6.2f} "
              f"{100*r['l1']:8.3f} {100*r['flux']:+11.3f} {100*r['maxpix']:10.2f}{tag}")

## 4. Convergence vs cost

The headline figure: relative L1 image difference (left) and absolute total-flux error (right) versus input spacing, one curve per spectrum, with the production 2 Å spacing marked. Coarser spacing is to the right and cheaper (cost ∝ 1/spacing). The gap between the continuum/physical curves and the narrow-line curve is the whole story — how much the answer depends on the input grid is set by the spectrum's own structure.

In [ ]:
fig, (axL, axF) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, rows in results.items():
    s = [r["spacing"] for r in rows]
    axL.plot(s, [100*r["l1"] for r in rows], "o-", label=name)
    axF.plot(s, [100*abs(r["flux"]) for r in rows], "o-", label=name)
for ax, ttl, yl in [(axL, "image L1 vs reference", "L1 difference [%]"),
                    (axF, "total-flux error vs reference", "|flux error| [%]")]:
    ax.axvline(2.0, color="0.6", ls="--", lw=1)
    ax.text(2.0, ax.get_ylim()[1], " 2 Å\n production", va="top", fontsize=8, color="0.4")
    ax.set(xscale="log", yscale="log", xlabel="input spacing Δλ [Å]", title=ttl, ylabel=yl)
    ax.set_xticks(SPACINGS); ax.set_xticklabels([f"{s:g}" for s in SPACINGS])
    ax.invert_xaxis()       # fine -> coarse (cheap) left -> right
    ax.grid(True, which="both", alpha=0.3)
axL.legend(fontsize=8)
fig.tight_layout()

## 5. The cost is linear in sample count

A quick confirmation that "samples" is the right cost axis: time the (warm) disperser at each spacing for the continuum case. The per-sample time is roughly constant, so wall-clock ∝ samples ∝ 1/Δλ — going from 2 Å to 1 Å really does roughly double the cost, and 2 Å → 4 Å roughly halves it.

In [ ]:
print(f"  {'Δλ[Å]':>6} {'samples':>8} {'time[ms]':>9} {'µs/sample':>10}")
for s in SPACINGS:
    wl_um, wl_a, dl = grid(s)
    c = counts_continuum(wl_um, wl_a, dl)
    disperse_counts(wl_um, c)                       # warm (compile is cached)
    t = time.time()
    for _ in range(3):
        disperse_counts(wl_um, c)
    ms = 1e3 * (time.time() - t) / 3
    print(f"  {s:6.2f} {wl_um.size:8d} {ms:9.1f} {1e3*ms/wl_um.size:10.2f}")

## 6. What a coarse grid does to a narrow line (image space)

To make the L1 numbers concrete, here is the FWHM = 2 Å line dispersed at a coarse 8 Å grid, the production 2 Å grid, and the 0.25 Å reference — the actual detector spot, plus its difference from the reference. As the input grid coarsens, the single sample landing on the line peak overcounts the narrow profile, so the spot brightens and its flux drifts from the converged value.

In [ ]:
xc = int(round(float(th.spectral_trace(opt1, X, Y, np.array([LAM0_A/1e4]))[0][0]))) - 1
yc = int(round(float(th.spectral_trace(opt1, X, Y, np.array([LAM0_A/1e4]))[1][0]))) - 1
H = 22
def spot(img): return img[yc-H:yc+H+1, xc-H:xc+H+1]

build2 = CASES["line FWHM 2 Å"]
wlr_um, wlr_a, dlr = grid(REF_DLAM)
ref_spot = spot(disperse_counts(wlr_um, build2(wlr_um, wlr_a, dlr)))

show = [8.0, 2.0, 0.5]
imgs = []
for s in show:
    wl_um, wl_a, dl = grid(s)
    imgs.append(spot(disperse_counts(wl_um, build2(wl_um, wl_a, dl))))
vmax = max(w.max() for w in imgs)

fig, axes = plt.subplots(2, len(show), figsize=(11, 7))
for j, (s, w) in enumerate(zip(show, imgs)):
    axes[0, j].imshow(w, origin="lower", cmap="inferno", vmin=0, vmax=vmax)
    axes[0, j].set(title=f"Δλ = {s:g} Å   (Σ={w.sum():.2f})", xticks=[], yticks=[])
    d = w - ref_spot; dl_ = np.abs(d).max() or 1.0
    im = axes[1, j].imshow(d, origin="lower", cmap="RdBu_r", vmin=-dl_, vmax=dl_)
    axes[1, j].set(title=f"minus 0.25 Å ref", xticks=[], yticks=[])
    fig.colorbar(im, ax=axes[1, j], fraction=0.046)
axes[0, 0].set_ylabel("line spot"); axes[1, 0].set_ylabel("difference")
fig.tight_layout()

## 7. Conclusions

How sensitive the dispersed image is to the input grid spacing depends entirely on the spectrum's structure relative to the grid:

- **Continuum and physical (KC96) sources: insensitive.** Their L1 difference is small already at 2 Å and grows only slowly toward coarse spacing, and the total flux barely drifts (a smooth/oversampled spectrum integrates well). For these, 2 Å is comfortably converged — and coarsening toward 4 Å for a ~2× speedup costs little image fidelity. The KC96 line is ~25 Å wide (template-limited), so it behaves like a continuum here.
- **Lines narrower than the grid: sensitive.** The FWHM 2 Å line's flux and image are badly mis-estimated once the spacing approaches or exceeds the line width (§3 table, §6 figure); it needs spacing comfortably finer than the line FWHM. The FWHM 5 Å line is far more forgiving.

**Takeaway for production.** 2 Å is a safe default for the spectra we actually disperse, because the catalog templates have no sub-grid lines — and the cost is linear in samples, so the spacing is a clean speed knob (§5). Use the worst-case *line width* present in your input spectra to pick the spacing: spacing well below the narrowest line you care about, and no finer. For the current continuum-and-broad-line templates, even 4 Å would be defensible; the limit is the input SED resolution, not the disperser.

*(Reproducibility: disperser pinned in `pixi.toml`; reference data version in `data/data-versions.lock`. The line cases are synthetic, centered on a grid point at every spacing; the KC96 case is the bundled starburst template at z=1.2, dispersed as a point source. Reference grid 0.25 Å.)*